In [ ]:
from pep_compass.local_enumeration.mutation.mutation_enumerator import (
    MutationEnumerationInTangentSpace,
)
from pep_compass.models.encoder_decoder.hydramp_encoder_decoder import (
    HydrAMPEncoderDecoder,
)
from pep_compass.models.encoder_decoder.utils import decoder_jacobian_approx

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
direction_threshold = 0.001
token_threshold = 0.1
mutang = MutationEnumerationInTangentSpace(
    direction_significance_threshold=direction_threshold,
    token_threshold=token_threshold,
)

hydramp = HydrAMPEncoderDecoder(
    jacobian_mode="approx",
    jacobian_eps=0.05,
    field_eps=0.05,
    device=device,
)

In [ ]:
hydramp.decoder_forward

In [ ]:
from functools import partial

dec_jac_wrpd = partial(
    decoder_jacobian_approx,
    decoder_forward=hydramp.decoder_forward,
    jacobian_eps=0.05,
)

In [ ]:
seq = "GMASKAGSVLGKITKIALGAL"
seq_tensor = hydramp.encode_peptides([seq])

jac = dec_jac_wrpd(x=seq_tensor)  # shape: (B, S*V, Z)
# --- SVD ---
U, S, Vh = torch.linalg.svd(jac, full_matrices=False)

In [ ]:
from typing import Optional


def filter_jac_dims_svd(
    *,
    jac: torch.Tensor,
    min_dim: Optional[int] = None,
    eigen_value_threshold: Optional[float] = None,
    trace_ratio_threshold: Optional[float] = None,
    cumulative_trace_ratio_threshold: Optional[float] = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Filter Jacobian dimensions using SVD-based criteria.

    Args:
        jac: Jacobian tensor of shape (B, S*V, Z)
        min_dim: Minimum number of dimensions to keep
        eigen_value_threshold: Keep dimensions with eigen values above this threshold
        trace_ratio_threshold: Keep dimensions with trace ratio above this threshold
        cumulative_trace_ratio_threshold: Keep dimensions until cumulative trace ratio exceeds this

    Returns:
        Filtered U, S, Vh tensors from SVD
    """
    assert not (
        eigen_value_threshold is not None
        and (
            cumulative_trace_ratio_threshold is not None
            or trace_ratio_threshold is not None
        )
    ), "Cannot use eigen_value_threshold with trace ratio thresholds"

    # --- SVD ---
    U, S, Vh = torch.linalg.svd(jac, full_matrices=False)
    n_dimensions = S.shape[1]

    if eigen_value_threshold is not None:
        n_dimensions = min(n_dimensions, (S > eigen_value_threshold).sum().item())
    elif (
        cumulative_trace_ratio_threshold is not None
        or trace_ratio_threshold is not None
    ):
        # Compute variance ratio from squared singular values
        trace_ratio = S.pow(2) / S.pow(2).sum(dim=1, keepdim=True)

        if trace_ratio_threshold is not None:
            n_dimensions = min(
                n_dimensions,
                (trace_ratio_threshold > trace_ratio_threshold).sum().item(),
            )
        if cumulative_trace_ratio_threshold is not None:
            cum_trace_ratio = trace_ratio.cumsum(dim=1)
            n_dimensions = min(
                n_dimensions,
                (cum_trace_ratio < cumulative_trace_ratio_threshold).sum().item() + 1,
            )

    if min_dim is not None:
        n_dimensions = max(n_dimensions, min_dim)

    return U[:, :, :n_dimensions], S[:, :n_dimensions], Vh[:, :n_dimensions, :]

---
unusful experiment

In [ ]:
# from typing import Optional


# def filter_jac_dims_svd(
#     *,
#     jac: torch.Tensor,
#     min_dim: Optional[int] = None,
#     singular_value_threshold: Optional[float] = None,
#     var_ratio_threshold: Optional[float] = None,
#     cumulative_var_ratio_threshold: Optional[float] = None,
# ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
#     assert not (
#         singular_value_threshold is not None
#         and (
#             cumulative_var_ratio_threshold is not None
#             or var_ratio_threshold is not None
#         )
#     )
#     # --- SVD ---
#     U, S, Vh = torch.linalg.svd(jac, full_matrices=False)
#     n_dimensions = S.shape[1]
#     if singular_value_threshold is not None:
#         n_dimensions = min(n_dimensions, (S > singular_value_threshold).sum())
#     elif cumulative_var_ratio_threshold is not None or var_ratio_threshold is not None:
#         # --- Center over the latent-dim (Z) ---
#         mu = jac.mean(dim=2, keepdim=True)  # shape: (B, S*V, 1)
#         jac_centered = jac - mu  # centered along latent dimension
#         # --- PCA via SVD ---
#         Upca, Spca, Vhpca = torch.linalg.svd(jac_centered, full_matrices=False)
#         var_ratio = Spca.pow(2) / Spca.pow(2).sum(dim=1, keepdim=True)
#         if var_ratio_threshold is not None:
#             n_dimensions = min(
#                 n_dimensions, (var_ratio < var_ratio_threshold).sum() + 1
#             )
#         if cumulative_var_ratio_threshold is not None:
#             cum_var_ratio = var_ratio.cumsum(dim=1)
#             n_dimensions = min(
#                 n_dimensions, (cum_var_ratio < cumulative_var_ratio_threshold).sum() + 1
#             )
#     if min_dim is not None:
#         n_dimensions = max(n_dimensions, min_dim)
#     return U[:, :, :n_dimensions], S[:, :n_dimensions], Vh[:, :n_dimensions, :]

In [ ]:
# U, S, Vh = filter_jac_dims_svd(jac=jac, singular_value_threshold=0.001)
# print(U.shape, S.shape, Vh.shape)
# U, S, Vh = filter_jac_dims_svd(jac=jac, cumulative_var_ratio_threshold=0.9)

In [ ]:
# U - orthonormal basis in the ambient tangent space
# S - singular values
# V - orthonormal basis in the latent tangent space

In [ ]:
# U.shape, S.shape, Vh.shape

In [ ]:
# adaptive variance based estimation of the dimensionality

In [ ]:
# geodesic based suggestion verification of the distance

In [ ]:
# check different eps of jacobian
# check stability of mutation suggestions

---
# curvature computation with christoffels

In [ ]:
import torch
from einops import rearrange
from pep_compass.geometry.utils import (
    approx_dg_from_jac,
    metric_from_jac,
    christoffel_from_jac_and_dg,
    riemann_from_Gamma_and_dGamma,
    ricci_from_riemann,
    scalar_from_ricci_and_metric,
    approx_dGamma_from_Gamma,
    log_map_shooting,
    exponential_map,
)

# ============================================================
# EXAMPLE: Compute curvature + geodesics for a peptide
#
# Assumes encode_peptides RETURNS LATENT z of shape (B, Z)
# And dec_jac_wrpd computes decoder Jacobian wrt latent.
# ============================================================

# ------------------------------
# 1. Encode peptide → LATENT VECTOR z
# ------------------------------
seq = "GMASKAGSVLGKITKIALGAL"
z = hydramp.encode_peptides([seq])  # THIS IS LATENT (B, Z), not tokens
print("Latent z shape:", z.shape)

In [ ]:
# ------------------------------
# 2. Compute decoder Jacobian at latent point
# ------------------------------
jac = dec_jac_wrpd(x=z)  # (B, M, Z)
B, M, Z = jac.shape
print("Jacobian:", jac.shape)

In [ ]:
# ------------------------------
# 3. Build perturbed latent positions: z + eps e_k
# ------------------------------
eps = 1e-3
eyeZ = torch.eye(Z, device=z.device)

x = z.clone()  # rename for clarity
x_pert = rearrange(x, "b z -> b 1 z") + eps * rearrange(eyeZ, "i j -> 1 i j")
x_pert_flat = rearrange(x_pert, "b k z -> (b k) z")

In [ ]:
# ------------------------------
# 4. Compute all perturbed Jacobians
# ------------------------------
jac_pert_flat = dec_jac_wrpd(x=x_pert_flat)
jac_pert = rearrange(jac_pert_flat, "(b k) m z -> b k m z", b=B, k=Z)
print("Perturbed Jacobians:", jac_pert.shape)

In [ ]:
# ------------------------------
# 5. Compute metric derivative
# ------------------------------
dg = approx_dg_from_jac(jac, jac_pert, eps)
print("dg:", dg.shape)

In [ ]:
# ------------------------------
# 6. Christoffel symbols Γ^k_{ij}
# ------------------------------
Gamma = christoffel_from_jac_and_dg(jac, dg)
print("Gamma:", Gamma.shape)

In [ ]:
# ------------------------------
# 7. Compute perturbed Christoffels (curvature requires Γ at perturbed points)
# ------------------------------
Gamma_pert_list = []
for k in range(Z):
    xk = x + eps * eyeZ[k]  # (B, Z)
    jac_k = dec_jac_wrpd(x=xk)  # (B, M, Z)

    # Build perturbations for xk
    xk_pert = rearrange(xk, "b z -> b 1 z") + eps * rearrange(eyeZ, "i j -> 1 i j")
    xk_pert_flat = rearrange(xk_pert, "b k z -> (b k) z")

    jac_k_pert_flat = dec_jac_wrpd(x=xk_pert_flat)
    jac_k_pert = rearrange(jac_k_pert_flat, "(b k) m z -> b k m z", b=B, k=Z)

    dg_k = approx_dg_from_jac(jac_k, jac_k_pert, eps)
    Gamma_k = christoffel_from_jac_and_dg(jac_k, dg_k)

    Gamma_pert_list.append(Gamma_k)

Gamma_pert = torch.stack(Gamma_pert_list, dim=1)  # (B, Z, Z, Z, Z)
print("Gamma_pert:", Gamma_pert.shape)

In [ ]:
# ------------------------------
# 8. Compute dGamma
# ------------------------------
dGamma = approx_dGamma_from_Gamma(Gamma, Gamma_pert, eps)
print("dGamma:", dGamma.shape)

In [ ]:
# ------------------------------
# 9. Riemann curvature, Ricci, Scalar Curvature
# ------------------------------
R = riemann_from_Gamma_and_dGamma(Gamma, dGamma)
print("Riemann:", R.shape)

Ric = ricci_from_riemann(R)
S = scalar_from_ricci_and_metric(Ric, metric_from_jac(jac))

print("Ricci:", Ric.shape)
print("Scalar curvature:", S)

In [ ]:
# ------------------------------
# 10. Exponential map
# ------------------------------
v = torch.randn_like(x)

In [ ]:
# constant Gamma approximation for demonstration
def Gamma_fn(_):
    return Gamma


xT = exponential_map(x, v, Gamma_fn, t1=0.0001)
print("xT:", xT)

# ------------------------------
# 11. Log map (shooting)
# ------------------------------
v_est = log_map_shooting(x, xT, Gamma_fn)
print("v_est:", v_est)

factor analysis in the latent

In [ ]:
# ============================================================
# Compare SVD vs Factor Analysis manifold directions
# ============================================================

import torch
import numpy as np
from einops import rearrange
from sklearn.decomposition import FactorAnalysis

# ------------------------------------------------------------
# 1. Prepare data matrix from Jacobian
# ------------------------------------------------------------
B, M, Z = jac.shape
X = rearrange(jac, "b m z -> (b m) z").detach().cpu().numpy()  # (N, Z)
X_centered = X - X.mean(axis=0, keepdims=True)

k = 10  # number of SVD/FA directions to compare

# ------------------------------------------------------------
# 2. SVD-based manifold directions (orthogonal)
# ------------------------------------------------------------
U_svd, S_svd, Vt_svd = np.linalg.svd(X_centered, full_matrices=False)
V_svd = Vt_svd.T[:, :k]  # (Z, k)

# normalize columns
V_svd /= np.linalg.norm(V_svd, axis=0, keepdims=True) + 1e-12

# ------------------------------------------------------------
# 3. Factor Analysis manifold directions (non-orthogonal)
# ------------------------------------------------------------
fa = FactorAnalysis(n_components=k)
fa.fit(X_centered)

Lambda = fa.components_.T  # (Z, k)
Lambda /= np.linalg.norm(Lambda, axis=0, keepdims=True) + 1e-12

# ------------------------------------------------------------
# 4. Cosine similarity matrix between SVD and FA directions
# ------------------------------------------------------------
C = V_svd.T @ Lambda  # (k, k)
print("Cosine similarity matrix (SVD vs FA):")
print(C)

# ------------------------------------------------------------
# 5. Principal angles between SVD and FA subspaces
# ------------------------------------------------------------
# Orthonormal basis for SVD is V_svd already
Q_svd = V_svd

# Orthonormal basis for FA subspace via QR
Q_fa, _ = np.linalg.qr(Lambda)
Q_fa = Q_fa[:, :k]

# Singular values of Q_svd^T Q_fa give cosines of principal angles
M_mat = Q_svd.T @ Q_fa
_, svals, _ = np.linalg.svd(M_mat)
angles = np.arccos(np.clip(svals, -1.0, 1.0))

print("\nPrincipal angles between SVD and FA subspaces (degrees):")
print(angles * 180 / np.pi)

# ------------------------------------------------------------
# 6. Stretching energies under decoder Riemannian metric
# ------------------------------------------------------------
g = metric_from_jac(jac)[0]  # (Z, Z) metric at the first point

V_svd_t = torch.from_numpy(V_svd).to(jac.device).float()  # (Z, k)
Lambda_t = torch.from_numpy(Lambda).to(jac.device).float()  # (Z, k)
g_t = g.float()

svd_energy = torch.einsum("zk,ij,zk->k", V_svd_t, g_t, V_svd_t)
fa_energy = torch.einsum("zk,ij,zk->k", Lambda_t, g_t, Lambda_t)


print("\nStretching energy along SVD directions:")
print(svd_energy)

print("\nStretching energy along FA directions:")
print(fa_energy)